# AsyncIO: Mastering Concurrent Python

This notebook covers the foundations of asynchronous programming in Python, focusing on `asyncio`. 

### Quick Comparison
- **AsyncIO**: For managing many **I/O-bound** tasks (e.g., API calls, database queries) using a single thread.
- **Multiprocessing**: For maximizing performance on **CPU-intensive** tasks by using multiple CPU cores.
- **Threading**: For parallel tasks that share memory but are still limited by the GIL (Global Interpreter Lock).

## 1. Core Concepts

| Concept | Description |
| :--- | :--- |
| **Coroutine** | A special function (`async def`) that can pause (`await`) and resume execution. |
| **Event Loop** | The central scheduler that manages and switches between coroutines. |
| **Awaitable** | Any object that can be used in an `await` expression (Coroutines, Tasks, Futures). |
| **Task** | A wrapper for a coroutine that schedules it to run on the event loop as soon as possible. |

> **The Chef Analogy:** Think of the event loop as a single chef. The chef starts boiling water (I/O wait). Instead of staring at the pot (blocking), the chef chops vegetables (switching tasks). When the water boils, the chef returns to it (event notification).

### ⚠️ The Golden Rule of AsyncIO
**Never use blocking calls** like `time.sleep()` or `requests.get()` inside `async` functions. They freeze the entire event loop. Use `await asyncio.sleep()` and `httpx.AsyncClient` instead.

### 💡 Note for Jupyter Users
Jupyter notebooks run inside an existing event loop. 
- **In a script:** Use `asyncio.run(main())` to start the loop.
- **In Jupyter:** Simply use `await main()` directly in a cell.

In [ ]:
import asyncio

async def say_hello():
    print("Hello...")
    await asyncio.sleep(1)
    print("...World!")

# In Jupyter, we await directly:
await say_hello()

## 2. Sequential vs. Concurrent Execution

Simply awaiting coroutines one after another is still sequential. To run them concurrently, you must wrap them in **Tasks** or use `asyncio.gather()`.

In [ ]:
async def fetch_data(id, delay):
    print(f"Task {id}: Fetching...")
    await asyncio.sleep(delay)
    print(f"Task {id}: Done!")
    return f"Data {id}"

async def sequential():
    # This takes 2 + 2 = 4 seconds
    res1 = await fetch_data(1, 2)
    res2 = await fetch_data(2, 2)
    return res1, res2

print("Running Sequentially:")
await sequential()

In [ ]:
async def concurrent():
    # This takes only 2 seconds total
    print("\nRunning Concurrently with gather:")
    results = await asyncio.gather(
        fetch_data(1, 2),
        fetch_data(2, 2)
    )
    return results

await concurrent()

## 3. Structured Concurrency (Python 3.11+)

`asyncio.TaskGroup` is the modern way to manage multiple tasks. If one task fails, it automatically cancels the others, preventing "zombie" tasks.

In [ ]:
async def main_tg():
    tasks = []
    async with asyncio.TaskGroup() as tg:
        for i in range(1, 4):
            task = tg.create_task(fetch_data(i, i))
            tasks.append(task)
    
    # Results are available after the block finishes
    return [t.result() for t in tasks]

results = await main_tg()
print(f"Results: {results}")

## 4. Real World: Concurrent HTTP Requests

Use `httpx` for asynchronous HTTP calls. It is much faster than `requests` for multiple endpoints.

In [ ]:
import httpx
import time

async def fetch_pokemon(client, pokemon_id):
    url = f"https://pokeapi.co/api/v2/pokemon/{pokemon_id}"
    resp = await client.get(url)
    return resp.json()["name"]

async def run_batch():
    async with httpx.AsyncClient() as client:
        start = time.perf_counter()
        # Schedule 20 requests concurrently
        tasks = [fetch_pokemon(client, i) for i in range(1, 21)]
        names = await asyncio.gather(*tasks)
        
        end = time.perf_counter()
        print(f"Fetched {len(names)} Pokémon in {end - start:.2f}s")
        print(f"First 5: {names[:5]}")

await run_batch()

## 5. Control Flow: Timeouts and Semaphores

- **Timeouts**: Prevent tasks from hanging forever.
- **Semaphores**: Limit the number of concurrent tasks (e.g., to avoid hitting API rate limits).

In [ ]:
sem = asyncio.Semaphore(2) # Only 2 at a time

async def limited_task(id):
    async with sem:
        print(f"Task {id} is working...")
        try:
            # Wrap with a 1-second timeout
            await asyncio.wait_for(asyncio.sleep(2), timeout=1.0)
        except asyncio.TimeoutError:
            print(f"Task {id} timed out!")

await asyncio.gather(*(limited_task(i) for i in range(5)))

## 6. Producer-Consumer Pattern (Queues)

Queues are perfect for decoupling data generation from data processing.

In [ ]:
async def producer(queue, n):
    for i in range(n):
        await asyncio.sleep(0.5)
        item = f"item-{i}"
        await queue.put(item)
        print(f"Produced {item}")

async def consumer(queue):
    while True:
        item = await queue.get()
        print(f"Consuming {item}...")
        await asyncio.sleep(1)
        queue.task_done()

async def run_queue():
    q = asyncio.Queue()
    # Start consumer in background
    cons = asyncio.create_task(consumer(q))
    
    await producer(q, 5)
    await q.join() # Wait for all items to be processed
    
    cons.cancel() # Stop the infinite consumer
    print("All work complete!")

await run_queue()